In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import make_scorer, matthews_corrcoef
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

# 1. Load and balance the dataset (339 of each class)
df = pd.read_csv('ai4i2020.csv')
feature_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]
X = df[feature_cols]
y = df["Machine failure"]

# Try imbalanced-learn undersampler, fallback to pandas sampling
try:
    from imblearn.under_sampling import RandomUnderSampler
    rus = RandomUnderSampler(sampling_strategy={0:339, 1:339}, random_state=42)
    X_bal, y_bal = rus.fit_resample(X, y)
except ImportError:
    print("imbalanced-learn not installed, sampling with pandas...")
    df_min = df[df['Machine failure'] == 1].sample(n=339, random_state=42)
    df_maj = df[df['Machine failure'] == 0].sample(n=339, random_state=42)
    df_bal = pd.concat([df_min, df_maj]).sample(frac=1, random_state=42)
    X_bal = df_bal[feature_cols]
    y_bal = df_bal['Machine failure']

# 2. Split into TRAIN (80%) and TEST (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_bal
)

# 3. Preprocessor: scale numeric features
preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), feature_cols)
])

# 4. Define MCC scorer
mcc_scorer = make_scorer(matthews_corrcoef)

# 5. Configure hyperparameter searches
search_configs = {
    "MLP": {
        "search": "random",
        "n_iter": 20,
        "pipeline": Pipeline([("scale", preprocessor), ("clf", MLPClassifier(max_iter=500, random_state=42))]),
        "params": {
            "clf__hidden_layer_sizes": [(50,), (100,), (50,50), (100,50)],
            "clf__activation": ["relu", "tanh", "logistic"],
            "clf__learning_rate": ["constant", "adaptive"],
            "clf__alpha": [1e-5, 1e-4, 1e-3]
        }
    },
    "SVM": {
        "search": "random",
        "n_iter": 20,
        "pipeline": Pipeline([("scale", preprocessor), ("clf", SVC(random_state=42))]),
        "params": {
            "clf__C": [0.1, 1, 10, 100],
            "clf__kernel": ["linear", "rbf", "poly"],
            "clf__gamma": ["scale", "auto"]
        }
    },
    "KNN": {
        "search": "random",
        "n_iter": 15,
        "pipeline": Pipeline([("scale", preprocessor), ("clf", KNeighborsClassifier())]),
        "params": {
            "clf__n_neighbors": [3, 5, 7, 9, 11],
            "clf__p": [1, 2],
            "clf__algorithm": ["auto", "ball_tree", "kd_tree"]
        }
    },
    "DecisionTree": {
        "search": "grid",
        "pipeline": Pipeline([("scale", preprocessor), ("clf", DecisionTreeClassifier(random_state=42))]),
        "params": {
            "clf__criterion": ["gini", "entropy"],
            "clf__max_depth": [None, 5, 10, 20],
            "clf__ccp_alpha": [0.0, 0.01, 0.05]
        }
    },
    "LogisticRegression": {
        "search": "grid",
        "pipeline": Pipeline([("scale", preprocessor), ("clf", LogisticRegression(random_state=42, solver="liblinear"))]),
        "params": {
            "clf__penalty": ["l2", "l1"],
            "clf__C": [0.01, 0.1, 1, 10, 100],
            "clf__solver": ["liblinear"]
        }
    }
}

# 6. Run hyperparameter tuning
best_estimators = {}
for name, cfg in search_configs.items():
    print(f"\n--- Tuning {name} ({cfg['search']} search) ---")
    if cfg["search"] == "random":
        searcher = RandomizedSearchCV(
            estimator=cfg["pipeline"],
            param_distributions=cfg["params"],
            n_iter=cfg["n_iter"],
            scoring=mcc_scorer,
            cv=5,
            random_state=42,
            n_jobs=-1,
            verbose=1
        )
    else:
        searcher = GridSearchCV(
            estimator=cfg["pipeline"],
            param_grid=cfg["params"],
            scoring=mcc_scorer,
            cv=5,
            n_jobs=-1,
            verbose=1
        )
    searcher.fit(X_train, y_train)
    print(f"{name} best CV MCC: {searcher.best_score_:.4f}")
    print(f"{name} best params: {searcher.best_params_}")
    best_estimators[name] = searcher.best_estimator_

# best_estimators now holds each tuned model ready for test-set evaluation.




--- Tuning MLP (random search) ---
Fitting 5 folds for each of 20 candidates, totalling 100 fits


/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/rod/MyApps/GitHub_Repos/CS_534_Intro_2_Artificial_Intelligence/group_project/Phase_2_Project_Proposal/mlp_model/.env/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't conver

MLP best CV MCC: 0.8232
MLP best params: {'clf__learning_rate': 'constant', 'clf__hidden_layer_sizes': (100, 50), 'clf__alpha': 0.001, 'clf__activation': 'relu'}

--- Tuning SVM (random search) ---
Fitting 5 folds for each of 20 candidates, totalling 100 fits
SVM best CV MCC: 0.8196
SVM best params: {'clf__kernel': 'rbf', 'clf__gamma': 'auto', 'clf__C': 100}

--- Tuning KNN (random search) ---
Fitting 5 folds for each of 15 candidates, totalling 75 fits
KNN best CV MCC: 0.7539
KNN best params: {'clf__p': 2, 'clf__n_neighbors': 7, 'clf__algorithm': 'ball_tree'}

--- Tuning DecisionTree (grid search) ---
Fitting 5 folds for each of 24 candidates, totalling 120 fits
DecisionTree best CV MCC: 0.8052
DecisionTree best params: {'clf__ccp_alpha': 0.0, 'clf__criterion': 'entropy', 'clf__max_depth': 5}

--- Tuning LogisticRegression (grid search) ---
Fitting 5 folds for each of 10 candidates, totalling 50 fits
LogisticRegression best CV MCC: 0.6320
LogisticRegression best params: {'clf__C': 10,